# 📓 **Lab 01 – Getting Hands‑On with AI Foundry Evaluations**  
Welcome to the first lab in the *AI Foundry Evaluation Series* 🎉  

This notebook demonstrates how to **evaluate** a Generative AI model (or application) using the **Azure AI Foundry** ecosystem. We'll highlight three key Python SDKs:
1. **`azure-ai-projects`** (`AIProjectClient`): manage & orchestrate evaluations in the cloud.
2. **`azure-ai-inference`**: perform model inference (optional but helpful if generating data for evaluation).
3. **`azure-ai-evaluation`**: run automated metrics for LLM output quality & safety.

We'll create or use some synthetic "health and life sciences" Q&A data, then measure how well your model is answering. We'll do both **local** evaluation and **cloud** evaluation (on an Azure AI Foundry project).

> **Disclaimer**: This covers a hypothetical health scenario. **No real medical advice** is provided. Always consult professionals.

## Notebook Contents
1. [Setup & Imports](#1-Setup-and-Imports)
2. [Local Evaluation Examples](#3-Local-Evaluation)
3. [Cloud Evaluation with `AIProjectClient`](#4-Cloud-Evaluation)
4. [Extra Topics](#5-Extra-Topics)
   - [Risk & Safety Evaluators](#5.1-Risk-and-Safety)
   - [More Quality Evaluators](#5.2-Quality)
   - [Custom Evaluators](#5.3-Custom)
   - [Simulators & Adversarial Data](#5.4-Simulators)
5. [Conclusion](#6-Conclusion)


## 1. Setup and Imports
We'll import necessary libraries, and define some synthetic data. 

### Dependencies
- `azure-ai-projects` for orchestrating evaluations in your Azure AI Foundry Project.
- `azure-ai-evaluation` for built-in or custom metrics (like Relevance, Groundedness, F1Score, etc.).
- `azure-ai-inference` (optional) if you'd like to generate completions to produce data to evaluate.
- `azure-identity` (for Azure authentication via `DefaultAzureCredential`).

### Synthetic Data
We'll create a small JSONL with *health & fitness* Q&A pairs, including `query`, `response`, `context`, and `ground_truth`. This simulates a scenario where we have user questions, the model's answers, plus a reference ground truth. 

> This format is equally important to maintain throughout test set generation, as it helps us to map values over to AI Foundry's cloud evaluation tool later.

In [1]:
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

import pandas as pd
import json
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import evaluate, ContentSafetyEvaluator
from azure.ai.projects import AIProjectClient
import os
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    Evaluation, Dataset, EvaluatorConfiguration, ConnectionType
)
from azure.ai.evaluation import F1ScoreEvaluator, RelevanceEvaluator, ViolenceEvaluator
from azure.identity import DefaultAzureCredential
from azure.core.exceptions import ServiceResponseError
import time

load_dotenv()

AZURE_AI_FOUNDRY_CONNECTION_STRING = os.getenv("AZURE_AI_FOUNDRY_CONNECTION_STRING")
_, AZ_SUBSCRIPTION_ID, AZ_RESOURCE_GROUP, AI_FOUNDRY_PROJECT_NAME  = AZURE_AI_FOUNDRY_CONNECTION_STRING.split(';')

azure_ai_project = {
    "subscription_id": AZ_SUBSCRIPTION_ID,
    "resource_group_name": AZ_RESOURCE_GROUP,
    "project_name": AI_FOUNDRY_PROJECT_NAME,
}

credential = DefaultAzureCredential()

ai_project_client = AIProjectClient.from_connection_string(
    credential=DefaultAzureCredential(),
    conn_str=AZURE_AI_FOUNDRY_CONNECTION_STRING,
)

print(f"Connected to Azure AI Project: {azure_ai_project['project_name']}")


Connected to Azure AI Project: medevals


Now to setup the synthetic data:

In [2]:
synthetic_eval_data = [
    {
        "query": "What is the recommended adult dose of acetaminophen per dose?",
        "context": "Maximum single dose for adults is 1000 mg every 4-6 hours, not to exceed 4000 mg in 24 hours.",
        "response": "500 mg every 6 hours",
        "ground_truth": "500–1000 mg every 4–6 hours, not exceeding 4000 mg per day"
    },
    {
        "query": "Common side effects of metformin?",
        "context": "Metformin often causes gastrointestinal symptoms.",
        "response": "Diarrhea and nausea",
        "ground_truth": "Diarrhea, nausea, abdominal discomfort"
    },
    {
        "query": "How often should a hypertensive patient check blood pressure?",
        "context": "Stable patients should monitor at least monthly.",
        "response": "Once a month",
        "ground_truth": "At least once a month or per clinician's advice"
    }
]

# Write them to a local JSONL file, this is a requirement for AI Foundry
eval_data_path = Path("./healthcare_eval_data.jsonl")
with eval_data_path.open("w", encoding="utf-8") as f:
    for row in synthetic_eval_data:
        f.write(json.dumps(row) + "\n")

print(f"Sample evaluation data written to {eval_data_path.resolve()}")

Sample evaluation data written to /Users/marcjimz/Documents/Development/aihlsignited-medevals/labs/healthcare_eval_data.jsonl


## 2. 🧪  Local Evaluation Examples

We'll now explore the shipped evaluators in `azure.ai.evaluation`. AI Foundry comes available with a set of built-in evaluators that can be used to evaluate the quality and safety of LLM outputs. It is important to realize that these evaluators are not specific to the health domain, but can be used for any domain, and that selecting one metric over another is a subjective choice. 

It is the culmination of the evaluation process that will determine the quality of the model, and not the individual metrics. You need to consider each and every evaluator in the context of your use case, and how they relate to each other. Consider these:

### Agent Evaluators
- **🎯 Intent Resolution**: Measures how well the agent identifies and clarifies user intent, including asking for clarifications and staying within scope.  
- **🛠️ Tool Call Accuracy**: Assesses the agent’s proficiency in selecting appropriate tools, and accurately extracting and processing inputs.  
- **✅ Task Adherence**: Evaluates how well the agent’s final response meets the predefined goal or request specified in the task.  
- **📝 Response Completeness**: Measures how comprehensive an agent’s response is when compared with the ground truth provided in a user’s input.  

### Retrieval‑Augmented Generation Evaluators
- **📚 Groundedness**: Measures how well the generated response aligns with the given context, focusing on its relevance and accuracy with respect to the context.  
- **🏅 Groundedness Pro**: Detects whether the generated text response is consistent or accurate with respect to the given context.  
- **🔍 Retrieval**: Evaluates the quality of search without ground truth, focusing on how relevant the context chunks are to address a query and how well the most relevant chunks are surfaced.  
- **🎯 Relevance**: Measures how effectively a response addresses a query, assessing accuracy, completeness, and direct relevance based solely on the given query.  

### General Evaluators
- **🔗 Coherence**: Measures the logical flow and organization of ideas in a response, allowing the reader to easily follow and understand the writer's train of thought.  
- **✍️ Fluency**: Assesses the clarity of written communication, focusing on grammatical accuracy, vocabulary range, sentence complexity, and overall readability.  

### Natural Language Comparison
- **🤝 Similarity**: Measures the semantic alignment between generated text and ground truth.  

### Traditional NLP Metrics
- **F1 Score**: Harmonic mean of precision and recall.  
- **BLEU**: Bilingual Evaluation Understudy for n‑gram overlap.  
- **GLEU**: Generalized BLEU optimized for sentence-level scoring.  
- **METEOR**: Metric for Evaluation of Translation with Explicit ORdering, accounts for synonymy and stemming.  
- **ROUGE**: Recall‑oriented Understudy for Gisting Evaluation (e.g., ROUGE‑L, ROUGE‑1/2).  

### Custom Evaluators
While the SDK provides a comprehensive set of built‑in evaluators for quality and safety, you may need to:
- Adapt grading rubrics (e.g., ignore HTML artifacts or structured headers).  
- Redefine evaluation criteria (e.g., factual correctness in groundedness).  
- Create entirely new evaluators tailored to your domain.

We recommend reviewing and extending our open‑source prompts and templates to build custom evaluators that align with your unique objectives, enabling transparent, human‑in‑the‑loop evaluation without the overhead of model fine‑tuning.


You can view all of these AI Foundry evaluators directly within AI Foundry, or within the Azure documentation. We will now want to:

We'll run local, code-based evaluation on the JSONL dataset we created. We'll:

1. **Load** the data.
2. **Define** one or more evaluators. (e.g. `F1ScoreEvaluator`, `RelevanceEvaluator`, `GroundednessEvaluator`, or custom.)
3. **Run** `evaluate(...)` to produce a dictionary of metrics.

> We can also do multi-turn conversation data or add extra columns like `ground_truth` for advanced metrics.

In [3]:
import os
from pathlib import Path
from azure.ai.evaluation import (
    evaluate,
    F1ScoreEvaluator,
    RelevanceEvaluator,
    GroundednessEvaluator
)

# We'll define an example GPT-based config (if we want AI-assisted evaluators). 
model_config = {
    "azure_endpoint": os.environ.get("AZURE_OPENAI_ENDPOINT"),
    "api_key": os.environ.get("AZURE_OPENAI_KEY"),
    "azure_deployment": os.environ.get(
        "AZURE_OPENAI_CHAT_DEPLOYMENT_ID"
    ),
    "api_version": os.environ.get("AZURE_OPENAI_API_VERSION"),
}

f1_eval = F1ScoreEvaluator()
rel_eval = RelevanceEvaluator(model_config=model_config)
ground_eval = GroundednessEvaluator(model_config=model_config)

# We'll run evaluate(...) with these evaluators.
results = evaluate(
    evaluation_name="healthcare-eval",
    data=str(eval_data_path),
    evaluators={
        "f1_score": f1_eval,
        "relevance": rel_eval,
        "groundedness": ground_eval,
    },
    azure_ai_project=None, #local
    evaluator_config={
        "f1_score": {
            "column_mapping": {
                "response": "${data.response}",
                "ground_truth": "${data.ground_truth}"
            }
        },
        "relevance": {
            "column_mapping": {
                "query": "${data.query}",
                "response": "${data.response}"
            }
        },
        "groundedness": {
            "column_mapping": {
                "context": "${data.context}",
                "response": "${data.response}"
            }
        },
        "resp_len": {
            "column_mapping": {
                "response": "${data.response}"
            }
        }
    }
)

[2025-04-23 09:34:37 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-23 09:34:37 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-23 09:34:37 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-23 09:34:37 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run azure_ai_evaluation_evaluators_common_base_eval_asyncevaluatorbase_8ixxn3ic_20250423_093437_763262, log path: /Users/marcjimz/.promptflow/.runs/azure_ai_evaluation_evaluators_common_base_eval_asyncevaluatorbase_8ixxn3ic_20250423_093437_763262/logs.txt
[2025-04-23 09:34:37 -0700][promptflow._sdk._orchestrator.

2025-04-23 09:34:38 -0700    2734 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-04-23 09:34:38 -0700    2734 execution.bulk     INFO     Finished 3 / 3 lines.
2025-04-23 09:34:38 -0700    2734 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.
======= Run Summary =======

Run name: "azure_ai_evaluation_evaluators_f1_score_f1_score_asyncf1scoreevaluator_swpykxqd_20250423_093437_762848"
Run status: "Completed"
Start time: "2025-04-23 09:34:37.759197-07:00"
Duration: "0:00:01.279414"
Output path: "/Users/marcjimz/.promptflow/.runs/azure_ai_evaluation_evaluators_f1_score_f1_score_asyncf1scoreevaluator_swpykxqd_20250423_093437_762848"

2025-04-23 09:34:40 -0700    2734 execution.bulk     INFO     Finished 1 / 3 lines.
2025-04-23 09:34:40 -0700    2734 execution.bulk     INFO     Average execution time for completed lines: 2.07 seconds. Es

Note how we had to generate a column mapping of the dataset to the evaluator. This is important to ensure that the evaluator can find the right columns in the dataset. This is made infinitely easier if our dataset adheres to the same columns required in AI Foundry, but in the event the dataset does not, we can always change the columns to the evaluator.

Let's look at what is now processed as a result of our evaluation:

In [4]:
print("Local evaluation result =>")
print(json.dumps(results['rows'][0], indent=3))

Local evaluation result =>
{
   "inputs.query": "What is the recommended adult dose of acetaminophen per dose?",
   "inputs.context": "Maximum single dose for adults is 1000 mg every 4-6 hours, not to exceed 4000 mg in 24 hours.",
   "inputs.response": "500 mg every 6 hours",
   "inputs.ground_truth": "500\u20131000 mg every 4\u20136 hours, not exceeding 4000 mg per day",
   "outputs.f1_score.f1_score": 0.375,
   "outputs.relevance.relevance": 3,
   "outputs.relevance.gpt_relevance": 3,
   "outputs.relevance.relevance_reason": "The response provides a relevant dosage but omits the maximum daily dosage, making it incomplete.",
   "outputs.groundedness.groundedness": 4,
   "outputs.groundedness.gpt_groundedness": 4,
   "outputs.groundedness.groundedness_reason": "The RESPONSE is missing critical details from the CONTEXT, such as the maximum single dose and the total daily limit, making it incomplete."
}


The results are now in a dictionary format, and we can use them to visualize the results. Note that everything that goes in as input is included, as well as everything on the output. Some of the fields are extra such as reasoning and results, and some are the actual metrics that were calculated. Not all of these are metrics, but they are all useful to understand how the evaluation was performed. We can pick and choose which ones of these we wish to use in the future for benchmarking of our application system performance.

# 3. Cloud Evaluation with `AIProjectClient`

Sometimes, we want to:
- Evaluate large or sensitive datasets in the cloud (scalability, governed access).
- Keep track of evaluation results in an Azure AI Foundry project.
- Push the execution of evaluation from our local machines and remote machines and have cloud manage it for us.

In this section, we'll demonstrate how to run an evaluation in the cloud using the `AIProjectClient`. This will allow us to leverage Azure's infrastructure for large-scale evaluations and keep track of results in our project.

We'll do that by:
1. **Upload** the local JSONL to your Azure AI Foundry project.
2. **Create** an `Evaluation` referencing built-in or custom evaluator definitions.
3. **Poll** until the job is done (with retry logic for resilience).
4. **Review** the results in the portal or via `project_client.evaluations.get(...)`.

### Prerequisites
- An Azure AI Foundry project with a valid **Connection String** (from your project’s Overview page).
- An Azure OpenAI deployment (if using AI-assisted evaluators).


In [5]:
# 1) Upload data for evaluation
uploaded_data_id, _ = ai_project_client.upload_file(str(eval_data_path))
print("✅ Uploaded JSONL to project. Data asset ID:", uploaded_data_id)

# 2) Create a evaluation object
evaluation = Evaluation(
    display_name="Foundry Remote Evaluation",
    description="Evaluating dataset for correctness.",
    data=Dataset(id=uploaded_data_id),
    evaluators={
        "f1_score": EvaluatorConfiguration(id=F1ScoreEvaluator.id),
        "relevance": EvaluatorConfiguration(
            id=RelevanceEvaluator.id,
            init_params={"model_config": model_config}
        ),
        "violence": EvaluatorConfiguration(
            id=ViolenceEvaluator.id,
            init_params={"azure_ai_project": ai_project_client.scope}
        )
    }
)

# Helper: Create evaluation with retry logic
def create_evaluation_with_retry(project_client, evaluation, max_retries=3, retry_delay=5):
    for attempt in range(max_retries):
        try:
            result = project_client.evaluations.create(evaluation=evaluation)
            return result
        except ServiceResponseError as e:
            if attempt == max_retries - 1:
                raise
            print(f"⚠️ Attempt {attempt+1} failed: {str(e)}. Retrying in {retry_delay} seconds...")
            time.sleep(retry_delay)

# 3) Create & track the evaluation using retry logic
cloud_eval = create_evaluation_with_retry(ai_project_client, evaluation)
print("✅ Created evaluation job. ID:", cloud_eval.id)

# 4) Poll or fetch final status
fetched_eval = ai_project_client.evaluations.get(cloud_eval.id)
print("Current status:", fetched_eval.status)
if hasattr(fetched_eval, 'properties'):
    link = fetched_eval.properties.get("AiStudioEvaluationUri", "")
    if link:
        print("View details in Foundry:", link)
else:
    print("No link found.")

Traceback (most recent call last):
  File "/Users/marcjimz/.pyenv/versions/list/lib/python3.13/site-packages/azure/monitor/opentelemetry/_configure.py", line 222, in _setup_instrumentations
    instrumentor: BaseInstrumentor = entry_point.load()
                                     ~~~~~~~~~~~~~~~~^^
  File "/Users/marcjimz/.pyenv/versions/list/lib/python3.13/site-packages/importlib_metadata/__init__.py", line 189, in load
    module = import_module(match.group('module'))
  File "/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python3.13/importlib/__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._

✅ Uploaded JSONL to project. Data asset ID: /subscriptions/28d2df62-e322-4b25-b581-c43b94bd2607/resourceGroups/rg-autoauth-eastus2-autoauth-ui-fix/providers/Microsoft.MachineLearningServices/workspaces/medevals/data/f2d59f2d-a041-45c9-aac2-0db0552013cf/versions/1
✅ Created evaluation job. ID: 7f15dcc8-3948-4153-ac4d-be45498c278f
Current status: Starting
View details in Foundry: https://ai.azure.com/build/evaluation/7f15dcc8-3948-4153-ac4d-be45498c278f?wsid=/subscriptions/28d2df62-e322-4b25-b581-c43b94bd2607/resourceGroups/rg-autoauth-eastus2-autoauth-ui-fix/providers/Microsoft.MachineLearningServices/workspaces/medevals


### Viewing Cloud Evaluation Results

You can click the URL generated above, give it a few minutes to generate as the remote submission takes time to generate a working URL as the evaluation is being polled. The URL takes you directly to the evaluation where you can:

1. View the evaluation status and metrics.
2. View row-level details and aggregated metrics.
3. View visualizations and insights that you can dynamically create.

# 4. AI Foundry Evaluation with Custom Evaluators

Not always do we have the luxury of using the built-in evaluators. Sometimes, we need to create our own custom evaluators. This is where the `azure-ai-evaluation` library comes in handy to specify the custom evaluations we want to build.

We are now going to build a custom evaluator using some simple Python code and importing packages that will help with evaluating string similarities. This custom evaluator could similarly be used to import models from HuggingFace and build semantic similarity with a model of your choosing, or other third party evaluators that you may wish to use. You can also consider custom logic and nuanced decision making with your own LLM-as-a-judge. 

The AI Foundry Evaluations framework makes it easy to extend and import the libraries of your choosing.

## Define Custom Evaluator

```python
from dataclasses import dataclass

from rapidfuzz import fuzz

from src.evals.custom.custom_evaluator import CustomEvaluator
from src.utils.ml_logging import get_logger

# Define a dataclass for returning fuzzy (indel) similarity
@dataclass
class IndelSimilarity:
    indel_similarity: float


class SlidingFuzzyEvaluator(CustomEvaluator):
    def __init__(self, **kwargs):
        """
        Initialize the evaluator with any number of keyword arguments.

        All keyword arguments provided during initialization are set as attributes of the instance.

        Example:
            evaluator = SlidingFuzzyEvaluator(param1="value1", param2=42)
            print(evaluator.param1)  # Output: "value1"
            print(evaluator.param2)  # Output: 42

        Parameters:
            **kwargs: Arbitrary keyword arguments.
        """
        super().__init__(**kwargs)
        self.logger = get_logger()
        for key, value in kwargs.items():
            setattr(self, key, value)

    def _clean_text(self, text: str) -> str:
        """
        Removes newline and tab characters from the given text.

        Parameters:
            text (str): The text to clean.

        Returns:
            str: The cleaned text.
        """
        # Replace newline and tab with a space, then collapse multiple spaces.
        return " ".join(text.replace("\n", " ").replace("\t", " ").split())

    def _sliding_window_fuzzy_match(self, ground_truth: str, response: str, step: int = 1) -> tuple:
        """
        Computes the best fuzzy matching score between the ground truth and any substring of the response
        using a sliding window of the same length as the ground truth.

        The window moves by the specified number of characters (step) each iteration.

        Parameters:
            ground_truth (str): The reference text.
            response (str): The larger text (e.g., OCR output) where the ground truth is searched.
            step (int): The number of characters to shift the window at each iteration (default is 1).

        Returns:
            tuple: (best_score, best_window, best_index)
                best_score (float): The highest fuzzy matching score.
                best_window (str): The substring of the response that produced the best score.
                best_index (int): The starting index of the best matching substring in the response.
        """
        window_size = len(ground_truth)
        best_score = 0
        best_window = ""
        best_index = -1

        # Slide through the response by the given step size (number of characters)
        for i in range(0, len(response) - window_size + 1, step):
            window = response[i:i + window_size]
            score = fuzz.ratio(ground_truth, window)
            if score > best_score:
                best_score = score
                best_window = window
                best_index = i
                # Early exit if a perfect match is found.
                if best_score == 100:
                    break
        return best_score, best_window, best_index

    def __call__(self, *, response: str, ground_truth: str, **kwargs) -> IndelSimilarity:
        """
        Computes fuzzy similarity between the response and ground_truth using a sliding window approach.
        Newline and tab characters are removed from the ground truth prior to matching.

        The sliding window moves by a number of characters defined by the 'step' keyword argument (default is 1).

        Signature:
            __call__(*, response: str, ground_truth: str, **kwargs) -> IndelSimilarity

        Returns:
            An IndelSimilarity instance containing the computed fuzzy (indel) similarity.
        """
        try:
            # Clean the ground truth to remove newline and tab characters.
            clean_ground_truth = self._clean_text(ground_truth)
            # Retrieve a custom step if provided, otherwise default to 10.
            step = kwargs.get("step", 10)
            best_score, best_window, best_index = self._sliding_window_fuzzy_match(
                clean_ground_truth, response, step=step
            )
            self.logger.info(f"Best score: {best_score} at index {best_index}.")
        except Exception as e:
            self.logger.error(f"Error computing similarity: {e}")
            best_score = 0

        return IndelSimilarity(indel_similarity=best_score)
```

In [6]:
# Import the Evaluator we created
from src.evals.custom.sliding_fuzzy_evaluator import SlidingFuzzyEvaluator

ModuleNotFoundError: No module named 'src'

We can now use this custom evaluator as part of our local evaluation run, observe as follows:

In [7]:
fuzzy_evaluator = SlidingFuzzyEvaluator()

# We'll run evaluate(...) with these evaluators.
results = evaluate(
    evaluation_name="healthcare-eval-semantic",
    data=str(eval_data_path),
    evaluators={
        "fuzzy_evaluator": SlidingFuzzyEvaluator,
    },
    azure_ai_project=None, #local
    evaluator_config={
        "fuzzy_evaluator": {
            "column_mapping": {
                "response": "${data.response}",
                "ground_truth": "${data.ground_truth}"
            }
        },
    }
)

[2025-04-23 09:23:00 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-23 09:23:00 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run azure_ai_evaluation_evaluators_fuzzy_evaluator_20250423_092300_467882, log path: /Users/marcjimz/.promptflow/.runs/azure_ai_evaluation_evaluators_fuzzy_evaluator_20250423_092300_467882/logs.txt


In [8]:
print("Local evaluation result =>")
print(json.dumps(results['rows'][0], indent=3))

Now for us to run this in cloud, we first must register them within our Azure AI Project and fetch the evaluator IDs. Here's how we can do it:

In [10]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from promptflow.client import PFClient

# Define ml_client to register custom evaluator
ml_client = MLClient(
       subscription_id=AZ_SUBSCRIPTION_ID,
       resource_group_name=AZ_RESOURCE_GROUP,
       workspace_name=AI_FOUNDRY_PROJECT_NAME,
       credential=DefaultAzureCredential()
)

# Then we convert the evaluator to evaluation flow and save it locally
pf_client = PFClient()
local_path = "sliding_fuzzy_evaluator_local"
pf_client.flows.save(entry=SlidingFuzzyEvaluator, path=local_path)

# Specify evaluator name to appear in the Evaluator library
evaluator_name = "SlidingFuzzyEvaluator"

# Finally register the evaluator to the Evaluator library
custom_evaluator = Model(
    path=local_path,
    name=evaluator_name,
    description="Evaluator calculating fuzzy similarity using the ground truth throughout the response. Great for considering matches against large body of texts using lexical similarity. ",
)
registered_evaluator = ml_client.evaluators.create_or_update(custom_evaluator)
print("Registered evaluator id:", registered_evaluator.id)
# Registered evaluators have versioning. You can always reference any version available.
versioned_evaluator = ml_client.evaluators.get(evaluator_name, version=1)
print("Versioned evaluator id:", registered_evaluator.id)

In [ ]:
print("test")

Now we can submit a cloud evaluation much like we did before, using this evaluator ID:

In [11]:
# 1) Create a evaluation object
evaluation_custom = Evaluation(
    display_name="Foundry Remote Evaluation - Custom Evaluator",
    description="Evaluating dataset for lexical similarity.",
    data=Dataset(id=uploaded_data_id),
    evaluators={
        "fuzzy_similarity": EvaluatorConfiguration(id=registered_evaluator.id),
    }
)

# 2) Create & track the evaluation using retry logic
cloud_eval = create_evaluation_with_retry(ai_project_client, evaluation_custom)
print("✅ Created evaluation job. ID:", cloud_eval.id)

# 4) Poll or fetch final status
fetched_eval = ai_project_client.evaluations.get(cloud_eval.id)
print("Current status:", fetched_eval.status)
if hasattr(fetched_eval, 'properties'):
    link = fetched_eval.properties.get("AiStudioEvaluationUri", "")
    if link:
        print("View details in Foundry:", link)
else:
    print("No link found.")

HttpResponseError: (UserError) Evaluation evaluator ID invalid asset or open AI grader format. Provided evaluator id is /subscriptions/28d2df62-e322-4b25-b581-c43b94bd2607/resourceGroups/rg-autoauth-eastus2-autoauth-ui-fix/providers/Microsoft.MachineLearningServices/workspaces/medevals/models/SlidingFuzzyEvaluator/versions/2 is invalid
Code: UserError
Message: Evaluation evaluator ID invalid asset or open AI grader format. Provided evaluator id is /subscriptions/28d2df62-e322-4b25-b581-c43b94bd2607/resourceGroups/rg-autoauth-eastus2-autoauth-ui-fix/providers/Microsoft.MachineLearningServices/workspaces/medevals/models/SlidingFuzzyEvaluator/versions/2 is invalid